In [1]:
# TASK 6

In [2]:
import pandas as pd
import numpy as np
import re

# ── 1. Load the raw sheet ────────────────────────────────────────────
df = pd.read_excel('textile_data.xlsx', sheet_name='RawData', skiprows=1)
df.columns = ['OrderID','Date','Region','Fabric_Type','Product',
              'Buyer_Type','Quantity_Meters','Revenue','Defect_Rate']
df.head()

,OrderID,Date,Region,Fabric_Type,Product,Buyer_Type,Quantity_Meters,Revenue,Defect_Rate
0,TX-2001,03-05-2023,NaN,silk,Garment Lot,Export - USA,4639.0,44348.84,NaN
1,TX-2002,03-03-2023,MULTAN,polyester,Curtain Cloth,Export - Europe,1522.0,14413.34,8.75
2,TX-2003,2024-03-04,sindh,COTTON,Shirt Fabric,Export - USA,4919.0,"PKR 14,215.91",3.24
3,TX-2004,11-14-2023,multan,DENIM,Curtain Cloth,Local Retailer,NaN,2114.24,10.46
4,TX-2005,21/05/2023,Faislabad,Poly-ester,Towel,Export - Europe,1289.0,10698.7,0.05


In [3]:
# Inspect missing values
print(df.isnull().sum())

OrderID             0
Date                4
Region              7
Fabric_Type         4
Product             5
Buyer_Type          0
Quantity_Meters     5
Revenue             6
Defect_Rate        10
dtype: int64


In [4]:
# Drop rows where the OrderID itself is missing (unrecoverable)
df = df.dropna(subset=['OrderID'])

# Fill categorical missing values with mode
for col in ['Region','Fabric_Type','Product','Buyer_Type']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Fill numeric missing values with median
for col in ['Quantity_Meters','Revenue','Defect_Rate']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

In [5]:
# Region – lowercase, strip, fix typos
region_map = {
    'punjab':'Punjab', 'PUNJAB':'Punjab', 'punjab ':'Punjab',
    'sindh':'Sindh', 'Sindh':'Sindh',
    'multan':'Multan', 'MULTAN':'Multan',
    'karachi':'Karachi', 'KARACHI':'Karachi',
    'lahore':'Lahore', 'lahor':'Lahore', 'Lahor':'Lahore',
    'faisalabad':'Faisalabad', 'faislabad':'Faisalabad',
    'Faislabad':'Faisalabad'
}
df['Region'] = df['Region'].astype(str).str.strip().map(
    lambda x: region_map.get(x.lower(), x))
df['Region'] = df['Region'].replace('nan', np.nan).fillna(df['Region'].mode()[0])

# Fabric Type – normalise case, fix variants
def normalise_fabric(x):
    x = str(x).strip().lower()
    if x in ['poly-ester','polyester','poly ester']:
        return 'Polyester'
    if x in ['linen','linnen']:
        return 'Linen'
    if x == 'cotton':
        return 'Cotton'
    if x == 'silk':
        return 'Silk'
    if x == 'denim':
        return 'Denim'
    return x.title()

df['Fabric_Type'] = df['Fabric_Type'].apply(normalise_fabric)

In [6]:
# Revenue – remove "PKR" prefix and commas
df['Revenue'] = df['Revenue'].astype(str).str.replace('PKR','',regex=False)\
                    .str.replace(',','',regex=False).str.strip()
df['Revenue'] = pd.to_numeric(df['Revenue'], errors='coerce')

# Defect Rate – strip '%' and convert
df['Defect_Rate'] = df['Defect_Rate'].astype(str).str.replace('%','',regex=False)\
                       .str.strip()
df['Defect_Rate'] = pd.to_numeric(df['Defect_Rate'], errors='coerce')

# Quantity – negative values are data-entry errors → make positive
df['Quantity_Meters'] = df['Quantity_Meters'].abs()

In [7]:
def parse_date(val):
    if pd.isna(val):
        return pd.NaT
    s = str(val).strip()
    # ISO format
    try: return pd.to_datetime(s, format='%Y-%m-%d')
    except: pass
    # Slash formats (DD/MM/YYYY or MM/DD/YYYY)
    if '/' in s:
        a,b,y = s.split('/')
        a,b,y = int(a), int(b), int(y)
        if a > 12:   return pd.to_datetime(f'{y}-{b:02d}-{a:02d}')
        if b > 12:   return pd.to_datetime(f'{y}-{a:02d}-{b:02d}')
        # ambiguous – assume DD/MM/YYYY (common in Pakistan)
        return pd.to_datetime(f'{y}-{b:02d}-{a:02d}')
    # Dash formats (MM-DD-YYYY)
    if '-' in s:
        a,b,y = s.split('-')
        a,b,y = int(a), int(b), int(y)
        if a > 12:   return pd.to_datetime(f'{y}-{b:02d}-{a:02d}')
        if b > 12:   return pd.to_datetime(f'{y}-{a:02d}-{b:02d}')
        return pd.to_datetime(f'{y}-{a:02d}-{b:02d}')
    return pd.NaT

df['Date'] = df['Date'].apply(parse_date)
df = df.dropna(subset=['Date'])

# Final dtypes
df['OrderID']     = df['OrderID'].astype(str)
df['Region']      = df['Region'].astype('category')
df['Fabric_Type'] = df['Fabric_Type'].astype('category')
df['Buyer_Type']  = df['Buyer_Type'].astype('category')
df['Quantity_Meters'] = df['Quantity_Meters'].astype(int)
df['Revenue']     = df['Revenue'].astype(float).round(2)
df['Defect_Rate'] = df['Defect_Rate'].astype(float).round(2)

In [8]:
# Remove fully blank rows
df = df.dropna(how='all')

# Remove duplicate OrderIDs (keep first)
df = df.drop_duplicates(subset='OrderID', keep='first')

print(f"Cleaned shape: {df.shape}")
df.head()

Cleaned shape: (60, 9)


,OrderID,Date,Region,Fabric_Type,Product,Buyer_Type,Quantity_Meters,Revenue,Defect_Rate
0,TX-2001,2023-03-05,Lahore,Silk,Garment Lot,Export - USA,4639,44348.84,6.26
1,TX-2002,2023-03-03,Multan,Polyester,Curtain Cloth,Export - Europe,1522,14413.34,8.75
2,TX-2003,2024-03-04,Sindh,Cotton,Shirt Fabric,Export - USA,4919,15416.95,3.24
3,TX-2004,2023-11-14,Multan,Denim,Curtain Cloth,Local Retailer,2778,2114.24,10.46
4,TX-2005,2023-05-21,Faisalabad,Polyester,Towel,Export - Europe,1289,10698.70,0.05


In [9]:
df.to_csv('textile_data_cleaned.csv', index=False)
print("Saved → textile_data_cleaned.csv")

Saved → textile_data_cleaned.csv
